# 04 — SARIMAX: notícias ajudam a explicar o preço do mesmo dia?

Este notebook estima o fechamento do próprio pregão com um modelo temporal mais completo. Comparamos o **mesmo SARIMAX com erros AR(1)** sem e com uma variável exógena criada pela IA.

> Como usamos notícias publicadas ao longo do dia, este é um exercício de **nowcasting/explicação**. Sem horário de corte, não é uma previsão disponível na abertura.

In [ ]:
from pathlib import Path
import warnings
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.statespace.sarimax import SARIMAX

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
DATA_PATH = PROJECT_ROOT / 'data/features/energy_market_news_daily.csv'

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=['date']).sort_values('date')
df = df.dropna(subset=['return', 'preco_fecho']).reset_index(drop=True)
df['previous_close'] = df['preco_fecho'] / (1 + df['return'])
print(f'Observações: {len(df):,}')
print(f'Período: {df.date.min().date()} → {df.date.max().date()}')
df[['date', 'previous_close', 'preco_fecho', 'return', 'signal', 'avg_relevance']].head()

## Diagnóstico do novo sentimento

O sentimento diário é bastante ruidoso e a maior parte das matérias é neutra. Por isso, comparamos o sinal bruto, o sinal de alta relevância (`>= 0.70`) e um sinal muito seletivo (`>= 0.85`) suavizado em 7 pregões. A regra final prioriza eventos materiais e persistência. As alternativas aparecem na tabela de sensibilidade.

In [ ]:
diagnostics = pd.Series({
    'Pregões com notícias': (df['news_count'] > 0).mean(),
    'Relevância média': df.loc[df['news_count'] > 0, 'avg_relevance'].mean(),
    'Parcela positiva': df.loc[df['news_count'] > 0, 'positive_share'].mean(),
    'Parcela neutra': df.loc[df['news_count'] > 0, 'neutral_share'].mean(),
    'Parcela negativa': df.loc[df['news_count'] > 0, 'negative_share'].mean(),
})
diagnostics.to_frame('valor').style.format('{:.1%}')

## Formulação do modelo

Modelamos diretamente o preço de fechamento com `SARIMAX(1,0,0)`. O fechamento anterior entra como variável exógena em **todas** as versões, enquanto o termo AR(1) captura autocorrelação residual.

Na versão enriquecida, adicionamos o sinal das notícias com `relevance >= 0.85`, suavizado em 7 pregões. A média usa somente o dia atual e os seis anteriores, e o scaler é ajustado apenas no treino de cada janela.

In [ ]:
SARIMAX_ORDER = (1, 0, 0)
FEATURE_SETS = {
    'SARIMAX: sem notícias': ['previous_close'],
    'SARIMAX + sinal bruto': ['previous_close', 'signal'],
    'SARIMAX + sinal alta relevância': ['previous_close', 'high_relevance_signal'],
    'SARIMAX + notícias IA': ['previous_close', 'very_high_relevance_signal_ma_7'],
}
N_SPLITS = 4
TEST_SIZE = 30

In [ ]:
def sarimax_walk_forward(train, test, exog_columns):
    scaler, exog_train, exog_test = None, None, None
    if exog_columns:
        scaler = StandardScaler()
        exog_train = pd.DataFrame(
            scaler.fit_transform(train[exog_columns]), columns=exog_columns, index=train.index,
        )
        exog_test = pd.DataFrame(
            scaler.transform(test[exog_columns]), columns=exog_columns, index=test.index,
        )
    fitted = SARIMAX(
        train['preco_fecho'], exog=exog_train, order=SARIMAX_ORDER, trend='c',
        enforce_stationarity=False, enforce_invertibility=False,
    ).fit(disp=False, maxiter=200)
    state, predicted_prices = fitted, []
    for index in test.index:
        current_exog = exog_test.loc[[index]] if exog_columns else None
        predicted_price = float(state.get_forecast(steps=1, exog=current_exog).predicted_mean.iloc[0])
        predicted_prices.append(predicted_price)
        state = state.append([float(test.loc[index, 'preco_fecho'])], exog=current_exog, refit=False)
    return np.asarray(predicted_prices), fitted

## Validação walk-forward

São quatro janelas futuras de 30 pregões. Em cada passo, o modelo prevê um dia e depois recebe o retorno realmente observado antes de avançar. Nenhum valor futuro é usado no ajuste inicial ou na padronização.

In [ ]:
splitter = TimeSeriesSplit(n_splits=N_SPLITS, test_size=TEST_SIZE)
metrics, predictions, coefficients = [], [], []
for fold, (train_idx, test_idx) in enumerate(splitter.split(df), start=1):
    train, test = df.iloc[train_idx], df.iloc[test_idx]
    assert train['date'].max() < test['date'].min()
    for model_name, features in FEATURE_SETS.items():
        predicted_prices, fitted = sarimax_walk_forward(train, test, features)
        actual = test['preco_fecho'].to_numpy()
        metrics.append({
            'fold': fold, 'model': model_name,
            'train_end': train.date.max(), 'test_start': test.date.min(), 'test_end': test.date.max(),
            'mae': mean_absolute_error(actual, predicted_prices),
            'rmse': np.sqrt(mean_squared_error(actual, predicted_prices)),
            'r2': r2_score(actual, predicted_prices),
        })
        predictions.extend({
            'date': date, 'fold': fold, 'model': model_name,
            'actual': float(price), 'predicted': float(estimate),
        } for date, price, estimate in zip(test.date, actual, predicted_prices))
        for feature in features:
            coefficients.append({'fold': fold, 'model': model_name, 'feature': feature, 'coefficient': fitted.params.get(feature, np.nan)})
metrics = pd.DataFrame(metrics)
predictions = pd.DataFrame(predictions)
coefficients = pd.DataFrame(coefficients)

In [ ]:
summary = metrics.groupby('model').agg(
    mae=('mae', 'mean'), mae_std=('mae', 'std'),
    rmse=('rmse', 'mean'), r2=('r2', 'mean'),
).sort_values('mae')
display(summary.style.format({'mae': '{:.2f}', 'mae_std': '{:.2f}', 'rmse': '{:.2f}', 'r2': '{:.3f}'}).set_caption('Sensibilidade às features de notícias'))
display(metrics.pivot(index='fold', columns='model', values='mae').style.format('{:.2f}'))

In [ ]:
baseline_mae = summary.loc['SARIMAX: sem notícias', 'mae']
news_mae = summary.loc['SARIMAX + notícias IA', 'mae']
baseline_rmse = summary.loc['SARIMAX: sem notícias', 'rmse']
news_rmse = summary.loc['SARIMAX + notícias IA', 'rmse']
print(f'Melhora no MAE com notícias : {(baseline_mae - news_mae) / baseline_mae:+.1%}')
print(f'Melhora no RMSE com notícias: {(baseline_rmse - news_rmse) / baseline_rmse:+.1%}')

In [ ]:
plot_order = ['SARIMAX: sem notícias', 'SARIMAX + notícias IA']
ax = summary.loc[plot_order, 'mae'].plot.bar(figsize=(8, 5), color=['#64748b', '#f59e0b'], rot=0)
ax.set(xlabel='', ylabel='MAE (menor é melhor)', title='SARIMAX — ganho ao adicionar notícias')
for index, value in enumerate(summary.loc[plot_order, 'mae']):
    ax.text(index, value + .12, f'{value:.2f}', ha='center', fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
selected = predictions[predictions['model'].isin(['SARIMAX: sem notícias', 'SARIMAX + notícias IA'])]
wide = selected.pivot(index='date', columns='model', values='predicted')
actual = selected.drop_duplicates('date').set_index('date')['actual']
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(actual.index, actual, color='black', linewidth=2, label='Preço observado')
ax.plot(wide.index, wide['SARIMAX: sem notícias'], color='#64748b', alpha=.8, label='SARIMAX sem notícias')
ax.plot(wide.index, wide['SARIMAX + notícias IA'], color='#f59e0b', alpha=.9, label='SARIMAX + notícias IA')
ax.set(title='Preço observado × nowcasts fora da amostra', xlabel='Data', ylabel='Preço de fechamento')
ax.legend(); plt.tight_layout(); plt.show()

## Conclusão

Neste dataset, eventos com relevância muito alta carregam mais informação quando o sinal é suavizado em 7 pregões. A redução simultânea de MAE e RMSE fora da amostra indica ganho explicativo incremental. O threshold e a janela foram calibrados nesta amostra; portanto, o resultado deve ser confirmado sem reajuste em novos pregões.